# Model Evaluation: Predicting Conflict Escalation

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B:** Baseline + Text embeddings

Text embeddings are integrated in four different ways:
* All events, no PCA (~790 features) `all_nopca`
* All events, PCA (~73 features) `all_pca`
* Conflict-only events, no PCA (~790 features) `conflict_nopca`
* Conflict-only events, PCA (~47 features) `conflict_pca`

This notebook compares the results from the best models where `k`=1.75 and the threshold fix has been applied, as dicussed in the methodology decisions notebook.

Each model configuration has a number of recorded results on:
* Train (2018-2022) - period of time where there is no civil war. Results on training-CV splits. 
* Onset (2023) - including the three months where civil war esclated (April 2023)
* Active (2024-2025) - full period of time with ongoing active civil war

In [ ]:
import pandas as pd
import plotly.express as px

from models.run_best_models import run_best_models
from utils.reporting import read_model_reports

MODELS = [
    "Model A",
    "Model B (conflict-only text PCA)",
    "Model B (conflict-only text non-PCA)",
    "Model B (all-event text non-PCA)",
    "Model B (all-event text PCA)",
]

In [1]:
def apply_final_config(df: pd.DataFrame, config) -> pd.DataFrame:
    """Filter the results for the decided config."""
    mask = pd.Series(True, index=df.index)
    for col, val in config.items():
        mask &= df[col] == val

    food_ok = (df["include_food"] == False) | (df["price_recency"] == True)
    mask &= food_ok
    return df[mask].copy()


part1_config = {
    "k": 1.75,
    "threshold_fix_applied": True,
}


all_results = pd.read_csv("evaluation/sudan_results.csv")
results = apply_final_config(all_results, part1_config)

NameError: name 'pd' is not defined

In [2]:
def variant_label(row):
    if row["include_text"] == False:
        return "model_a"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == True:
        return "conflict_pca"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == False:
        return "conflict_nopca"
    if row["conflict_only_embeddings"] == False and row["use_pca"] == True:
        return "all_pca"
    return "all_nopca"


def baseline_comparison_table(results, metrics):

    for m in metrics:
        results[m] = pd.to_numeric(results[m], errors="coerce")

    # Columns that define a "matched" comparison so it differs only when text is included
    key_cols = [
        "include_food",
        "include_rain",
        "k",
        "event_col",
        "n_splits",
        "price_recency",
    ]

    baseline = results[results["include_text"] == False].set_index(
        key_cols
    )  # Baseline - no-text

    text_df = results[results["include_text"] == True].copy()  # Text runs

    text_df["variant"] = text_df.apply(variant_label, axis=1)

    results_comparison = []
    for variant, grp in text_df.groupby("variant"):
        grp = grp.set_index(key_cols)
        matched = grp.join(baseline[metrics], rsuffix="_base", how="inner")

        # Calculate average n_predictors for the variant
        mean_preds = (
            matched["n_predictors"].mean()
            if "n_predictors" in matched.columns
            else float("nan")
        )

        row = {
            "variant": variant,
            "n_matched_pairs": len(matched),
            "mean_n_predictors": round(mean_preds, 1),
        }

        for m in metrics:
            diff = matched[m] - matched[f"{m}_base"]
            row[f"percent_better_baseline_{m}"] = round((diff > 0).mean() * 100, 1)
            row[f"mean_diff_{m}"] = round(diff.mean(), 4)
            row[f"median_diff_{m}"] = round(diff.median(), 4)

        results_comparison.append(row)

    results_table = pd.DataFrame(results_comparison).set_index("variant")
    display(results_table)

# Part 1 - does text improve model performance?
Part 1 looks at the question of 'does text improve performance?' rather than comparing the final chosen best models. The following things have been fixed (see methdodology decisions for discussion):

* `k`=1.75 - this is fixed as it defines the prediction target.
* The threshold fix has been applied - this was a bug fix that has been resolved. 
* Food price recency flag has been included - this was a data porcessing issue that has been resolved.

The following implementation choices are allowed to roam in part 1 as they are do not change the underlying task, only what inputs the model draws on and how it is fit:
* `n_splits`
* `event_col`
* The inclusion of additional strucutral variables (food/rain)

# 1.1 Training performance
The results on the cross-validation training splits give an indication on whether the models are overfitting to the training data and would therefore not be generalisable.

As the aim is to understand if adding text improves the baseline (structural features only), each baseline and text-added configuration was compared on training-CV AUPR and F1. The percentage of times that text improved each metric is recorded in the table below. The headline numbers indicate that text hurts model performance in training: on AUPR, only two of the four variants ever beat baseline, and only barely (four models total).

Conflict-only text, both with (`conflict_pca`) and without PCA (`conflict_nopca`), is tied as the best performing model on training AUPR, each beating baseline in 2 of 16 comparisons (12.5%). Neither all-event (`all_nopca` and `all_pca`) variant beats baseline on AUPR in any comparison. On F1, conflict-only text without PCA (`conflict_nopca`) is the clearer of the two, beating baseline in 4 of 16 comparisons (25%) against conflict-only PCA's 2 of 16 (12.5%), so it's the stronger overall performer of the two once both metrics are considered, but the AUPR result on its own does not distinguish between them.

In [ ]:
train_metrics = ["train_cv_aupr", "train_cv_f1"]
baseline_comparison_table(results, train_metrics)

# 1.2 Onset performance

Unlike training, text clearly helps at onset, but which text variant looks best depends on which metric is prioritised, and that split lines up with the text corpus each model draws on, not with PCA.

Both conflict-only variants outperform both all-event variants on AUPR:
* Conflict-only without PCA (`conflict_nopca`): 75.0% of 16 matched pairs beat baseline. With a mean AUPR diff of +0.017.
* Conflict-only with PCA (`conflict_pca`): 68.8% of 16 models beat baseline. With +0.009 mean AUPR diff
In this model's context, AUPR demonstrates how well the model balances the accuracy of the region-months it flags as escalations (precision) and its ability to find the actual escalations (recall) across all confidence thresholds.

In contrast both all-event variants outperform both conflict-only variants on F1 (all-event non-PCA 87.5%, mean diff +0.032; all-event PCA 43.8%, mean diff -0.005). All-event text with PCA is the weakest text variant on AUPR specifically. It is  the only one with a negative mean difference (-0.0085), and the only one that fails to beat baseline in a majority of comparisons on either metric.

This isn't two variants pulling in different directions by chance. Conflict-only text runs a much more conservative operating point, it ranks escalation months well overall (hence the stronger AUPR across both its PCA and non-PCA forms), but is cautious about calling any specific month an escalation. All-event text runs the opposite, flagging more broadly, which drives its F1 advantage but also means two of all-event non-PCA's sixteen configurations collapse to predicting esclations for almost everything (`onset_recall_class1 > 0.9`). Given that a missed escalation is treated as the more costly error for an early-warning system in this project's framing, all-event text's recall-leaning approach is not obviously the wrong choice, but it should be considered a trade-off.

In [ ]:
onset_metrics = ["onset_aupr", "onset_f1_class1"]
baseline_comparison_table(results, onset_metrics)

Both all-event text variants sit above baseline on recall and below it on precision. Both conflict-only text variants sit below baseline on recall and above it on precision. 

`all_nopca` is the extreme case at both ends. It has nearly double baseline's recall, and the only variant with a high collapse rate (12.5%) when setting the collapse threshold to `recall > 0.9`. `all_pca` shows recall in the same direction but far more mildly (recall 0.425 vs baseline's 0.384, a modest lean rather than a strong one), and its 6.2% collapse rate (1 of 16 configs) it noteworthy but not the dominant outcome. 

In [ ]:
def onset_recall_precision_summary(results):
    results = results.copy()
    for col in ["onset_recall_class1", "onset_precision_class1"]:
        results[col] = pd.to_numeric(results[col], errors="coerce")

    results["variant"] = results.apply(variant_label, axis=1)
    results["collapsed"] = results["onset_recall_class1"] > 0.9

    return (
        results.groupby("variant")
        .agg(
            n_configs=("onset_recall_class1", "count"),
            mean_recall=("onset_recall_class1", "mean"),
            mean_precision=("onset_precision_class1", "mean"),
            collapse_rate=("collapsed", "mean"),
        )
        .round(3)
    )

onset_recall_precision_summary(results)


# 1.3 Active performance

Once the war is underway, the baseline model prevails almost everywhere. All four text variants now show negative mean AUPR differences against baseline.

Three of the four text variants perform very poorly during active conflict, and all show negative mean differences on both AUPR and F1. Once conflict has been running for months, region-month event counts stop being zero-inflated and the autoregressive/structural features are doing the actual work. The text embeddings aren't acting as a precursor signal the way they might pre-escalation, they're mostly extra dimensions that can cause overfitting.

During active conflict, the text model all-text with PCA is the best performing against baseline. During active conflict dimensionality reduction appears to specifically help text remain useful once conflict is underway. However, the same compression only works slightly on the conflict-only text model during active war. The pattern suggests it isn't PCA alone or text alone driving the active-period result, but the combination of the full event corpus (not just conflict events) compressed down to a smaller, less overfitting-prone feature set.

In [ ]:
active_metrics = ["active_aupr", "active_f1_class1"]
baseline_comparison_table(results, active_metrics)

# Part 2
For part two, the single best model configuration for the baseline model (Model A) will be comapred against the best model configs for the text variants. In order to make a fair comparison, configurations that were allowed to run freely in part 1 have been limited.

# 2.1 Defining the final config

## 2.1.1 Setting included data

In part 1 dropping rain or dropping food, sometimes scores marginally higher for some variants. Deliberately not following that signal here is the point, if the final comparison quietly adopted whichever ablation flattered each model, the comparison would no longer be about text, it would be about whichever feature set happened to win, model by model. 

However for the final comparison between Model A and Model B variants, food and rain will be included in all. 

## 2.1.2 Setting n_splits

The `n_splits` parameter determines the number of expanding-window cross-validation folds used during hyperparameter tuning (testing both `n=4$`and `n=5` across models). 

Across models, cross-validation training AUPR scores between the two choices are almost identical, showing a small difference of under 0.006 (0.2220 for four splits versus 0.2162 for five splits). Five splits is selected because its structure aligns naturally with the five-year training period (2018–2022), providing intuitive annual expanding increments that maximise training data volume per fold. 

A supporting check (holding food, rain, and event_col fixed and varying only n_splits) found the two all-event text variants are 1.5-1.7x more
sensitive to this choice than the baseline, while conflict-only text and the baseline itself are comparatively stable. Results for all-event text in Part 2 should be read with that in mind.

In [ ]:
metrics = ["train_cv_aupr", "onset_aupr", "active_aupr"]

results.groupby("n_splits")[metrics].agg(["mean", "max", "count"]).round(4)

In [ ]:
results["variant"] = results.apply(variant_label, axis=1)
results.groupby(["n_splits", "variant"])[metrics].agg(["mean", "count"]).round(4)

## 2.1.3 Event type
`event_col` determines whether ACLED event features are chosen from the six events or the 25 sub-event types. Sub-events naturally give the model greater detail but this level of disaggregation reduces the number of positive instances of that sub-event. 

Interestingly three of five models (conflict-only non-PCA, conflict-only PCA, and Model A itself) differ in the best `event_type` depending on whether train-CV AUPR or onset AUPR is prioritised. 

To remain objective (rather than just picking the `event_type` that provides the best onset AUPR), `event_col` is selected using the column that most often increases train-CV AUPR, which is independent of the onset and active periods being reported on. 

`event_col` is set to sub_event_type throughout, the option train-CV AUPR narrowly favoured overall (21 of 40 paired comparisons).

In [ ]:
results_fixed_data = results[
    (results["include_food"] == True) & (results["include_rain"] == True)
]

for variant, group in results_fixed_data.groupby("variant"):
    g = group.set_index("event_col")[["train_cv_aupr", "onset_aupr"]]
    train_winner = g["train_cv_aupr"].idxmax()
    onset_winner = g["onset_aupr"].idxmax()
    agree = "AGREE" if train_winner == onset_winner else "DISAGREE"
    print(
        f"{variant}: train_cv prefers {train_winner}, onset prefers {onset_winner} -> {agree}"
    )

In [ ]:
def event_col_preference_table(results, metrics):
    match_cols = ["include_food", "include_rain", "n_splits", "variant"]

    sub = results[results["event_col"] == "sub_event_type"].set_index(match_cols)
    evt = results[results["event_col"] == "event_type"].set_index(match_cols)

    rows = []
    for metric in metrics:
        paired = sub[[metric]].join(
            evt[[metric]], lsuffix="_sub", rsuffix="_evt", how="inner"
        )
        for variant, grp in paired.reset_index().groupby("variant"):
            n_sub_wins = (grp[f"{metric}_sub"] > grp[f"{metric}_evt"]).sum()
            n_evt_wins = (grp[f"{metric}_evt"] > grp[f"{metric}_sub"]).sum()
            n_total = len(grp)
            rows.append(
                {
                    "variant": variant,
                    "metric": metric,
                    "n_pairs": n_total,
                    "sub_event_type_wins": n_sub_wins,
                    "event_type_wins": n_evt_wins,
                }
            )

    table = pd.DataFrame(rows)

    totals = table.groupby("metric")[
        ["n_pairs", "sub_event_type_wins", "event_type_wins"]
    ].sum()
    totals["variant"] = "TOTAL"
    totals = totals.reset_index()
    print(totals)

    return pd.concat([table, totals], ignore_index=True)


event_col_preference_table(results, ["train_cv_aupr", "onset_aupr"])

In [ ]:
set_confg = {
    "k": 1.75,
    "threshold_fix_applied": True,
    "price_recency": True,
    "event_col": "sub_event_type",
    "include_food": True,
    "include_rain": True,
    "n_splits": 5,
}

In [ ]:
# run_best_models(set_config) # No need to rerun 
best_model_results, best_model_params, best_model_shap, best_model_onset_pred = read_model_reports(MODELS)

# 2.2 Regional onset performance
Evaluating aggregate onset AUPR alone suggests that conflict-only text variants perform strongly. However, this aggregate metric masks critical geographic failures when broken down by region. Because the 2023 escalation was concentrated in specific hotspots, an early-warning system must successfully flag escalation in the regions where fighting actually broke out—most notably Khartoum in April 2023.  The table below evaluates performance using the best configuration for each variant, focusing on recall across key escalation regions and Khartoum specifically.  

This is run on the best config for each of the variants. 

In [ ]:
KEY_REGIONS = [
    "Khartoum", "North Darfur", "South Darfur", "West Darfur",
    "Central Darfur", "East Darfur", "West Kordofan", "South Kordofan",
]

def region_recall_table(onset_pred_df, regions, model_col="model"):
    rows = []
    for model, grp in onset_pred_df.groupby(model_col):
        key_region_rows = grp[grp["region"].isin(regions)]
        khartoum_rows = grp[grp["region"] == "Khartoum"]

        def recall_precision(sub):
            n_true = sub["y_true"].sum()
            n_pred_pos = sub["y_pred"].sum()
            n_caught = ((sub["y_true"] == 1) & (sub["y_pred"] == 1)).sum()
            recall = n_caught / n_true if n_true else float("nan")
            precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
            return n_caught, n_true, n_pred_pos, recall, precision

        n_caught, n_true, n_pred_pos, key_recall, key_precision = recall_precision(key_region_rows)
        kh_caught, kh_true, kh_pred_pos, kh_recall, kh_precision = recall_precision(khartoum_rows)

        overall_pred_pos_rate = grp["y_pred"].mean()

        rows.append({
            "model": model,
            "overall_pred_positive_rate": round(overall_pred_pos_rate, 3),
            "key_regions_caught": f"{n_caught}/{n_true}",
            "key_regions_recall": round(key_recall, 3),
            "key_regions_precision": round(key_precision, 3),
            "khartoum_caught": f"{kh_caught}/{kh_true}",
            "khartoum_recall": round(kh_recall, 3),
            "khartoum_precision": round(kh_precision, 3),
        })
    return pd.DataFrame(rows).set_index("model")

region_recall_table(best_model_onset_pred, KEY_REGIONS)


Conflict-only non-PCA isn't the best variant here. It misses all esclations in Khartoum entirely and only finds one of the 27 escalations. All-event text without PCA does the opposite: it beats both the baseline and conflict-only PCA on regional recall, including catching escalation months in Khartoum that neither of the other two models flag.

At first glance, all-event text without PCA has the highest regional and Khartoum recall of all models, but this needs reading against its overall predicted-positive rate, which is far higher than every other model's (recall 0.889 overall, against 0.14-0.55 for the other four models).  Its regional precision (0.286) is actually higher than its overall precision (0.240), however this sia pretty low level of precision and is the lowest overall. 

This is the same trade-off that shows up in the aggregate onset F1 numbers above (all-event text has the stronger F1, conflict-only PCA the stronger AUPR), just made concrete at the region level. AUPR is measuring ranking quality across every possible threshold, so a model can score well there while still, at the actual threshold it's deployed with, being too conservative to flag the events that matter most. The conflict-only PCA model's overalll onset recall sits well below the other two models (see Section 2), and that is most present in exactly the regions where the war started.

For an early-warning use case, missing Khartoum isn't a small issue, it is fundamentally failing to achieve its purpose. On that basis, all-event text without PCA is the better candidate for onset detection specifically, even though it's the weaker performer on train-CV and on aggregate AUPR. However, this is a very small sample size of actual escalation. This also points to a broader limitation of this sample size. The onset finding rests on a single historical case as Sudan's 2023 escalation is the only event of its kind in this dataset, so the claim that text helps at onset should be read as "text helped for this one escalation", not as a validated general property. 

# 2.3 Feature importance: why the pattern flips

SHAP scores help identify why the pattern changes. SHAP importance was computed separately on the training, onset, and active periods to test whether text's contribution shifts once conflict is underway. 

Text embeddings' share of importance does not shrink during active war, it increases in all four text variants (e.g. all-event non-PCA rises from 50.9% of importance in training to 58.9% during active conflict). Rather than the model correctly deprioritising text once its precursor value is no longer relevant, it appears to rely on text more heavily during exactly the period where that reliance coincides with the sharpest drop in predictive performance. This suggests the active-period AUPR and F1 declines are indicative of the model continuing to draw on text-derived signal that does not generalise to active-conflict dynamics. 

In [ ]:
shap_by_category = (
    best_model_shap.groupby(["model", "dataset", "category"])["mean_abs_shap"]
    .sum()
    .reset_index()
)

shap_by_category["total_SHAP"] = shap_by_category.groupby(["model", "dataset"])[
    "mean_abs_shap"
].transform("sum")
shap_by_category["% Importance"] = (
    shap_by_category["mean_abs_shap"] / shap_by_category["total_SHAP"] * 100
)

fig = px.bar(
    shap_by_category,
    x="model",
    y="% Importance",
    color="category",
    facet_row="dataset",
    category_orders={"dataset": ["train", "onset", "active"]},
    title="SHAP feature importance by category (Train vs Onset vs Active)",
    text_auto=".1f",
    color_discrete_sequence=px.colors.qualitative.Bold,
    height=900,
)

fig.update_layout(
    xaxis_title="",
    legend_title_text="Feature Category",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))

# Add light gridlines and standardise y-axis titles for all rows
fig.update_yaxes(
    title_text="Relative Importance (%)",
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black")

fig.show()

# 2.4. Ethiopia results to do

# 3. Summary

Pulling the findings together across all evaluation windows:

* **Train-CV:** None of the text models outperform the baseline in terms of generalisation during training (2018–2022). Conflict-only text without PCA is the least weak of four underperforming options, beating baseline on F1 in 25% of matched pairs and 12.5% on AUPR. This performance gap is plausibly explained by the training period containing no civil war dynamics to learn from.

* **Onset (Aggregate):** Performance splits cleanly along corpus lines. Conflict-only text scores well on aggregate AUPR (beating baseline in 68.8–75.0% of matched pairs) but fails on F1 (12.5–25.0%), indicating a conservative operating point that ranks escalation months well overall but is too cautious at the deployed threshold. Conversely, all-event text without PCA dominates on F1 (87.5% better than baseline), prioritising recall over precision.

* **Onset (Regional):** Aggregate AUPR conceals geographic failures. Conflict-only text variants fail in the real-life scenario, completely missing the Khartoum outbreak (0/5) and catching at most 4 of 27 key regional escalation months. All-event text without PCA is the only model that catches fighting where it actually occurred (24/27 key regions, 2/5 Khartoum), though this high recall comes at the cost of a lower precision level.

* **Active Conflict:** The baseline wins clearly for three of the four text variants once active war is underway, and decisively so for conflict-only non-PCA. All-event text with PCA is the sole exception, running roughly level with or marginally ahead of baseline, demonstrating that corpus breadth combined with dimensionality reduction is required to maintain signal during active conflict.


Given the project's objective is to evaluate whether adding text improves conflict escalation predictions, **all-event text without PCA** remains the strongest candidate for onset detection specifically, as it is the only variant that successfully flags real-world escalation hubs like Khartoum. For ongoing active conflict monitoring, **all-event text with PCA** is worth retaining, as it is the only text model that avoids performance decay once conflict is underway. Conflict-only text, in either PCA or non-PCA form, is the weakest performer across all three periods and is hardest to justify keeping in the final model.

# 4. Testing across different parameters

In [ ]:
seeds = pd.read_csv("evaluation/sudan_results_seeds.csv")
seeds["variant"] = seeds.apply(variant_label, axis=1)
metrics = ["onset_aupr", "onset_f1_class1", "active_aupr", "active_f1_class1"]

for m in metrics:
    seeds[m] = pd.to_numeric(seeds[m], errors="coerce")

# Aggregate performance across seeds
seed_summary = seeds.groupby("variant")[metrics].agg(["mean", "std", "min", "max"]).round(4)
print(seed_summary)